# 第4章：Reduce算子与优先队列模拟堆 — 动手实验

本 notebook 指导你完成3个 Ascend C 自定义算子的编译、运行和验证。

**实验目录**：`src/reduce_lab/`

## 1. 检查实验环境

在运行实验之前，需要先确认 CANN SDK 和 ReduceLab 工程目录已经正确加载。下面的代码会检查 CANN 环境变量、目标平台和实验目录是否存在。

In [ ]:
import os

ASCEND_HOME = os.environ.get('ASCEND_HOME_PATH', '/home/developer/Ascend/cann-9.0.0')
assert os.path.exists(ASCEND_HOME), f'CANN SDK 未安装: {ASCEND_HOME}'
print(f'CANN SDK 路径: {ASCEND_HOME}')

# 目标平台：ascend310b 或 ascend910b
TARGET = os.environ.get('TARGET', 'ascend910b')
print(f'目标平台: {TARGET}')

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
if not os.path.exists(os.path.join(NOTEBOOK_DIR, 'src', 'reduce_lab')):
    NOTEBOOK_DIR = os.getcwd()
LAB_DIR = os.path.join(NOTEBOOK_DIR, 'src', 'reduce_lab')
assert os.path.exists(LAB_DIR), f'实验目录不存在: {LAB_DIR}'
print(f'实验目录: {LAB_DIR}')
print('环境检查通过')

输出"环境检查通过"说明 Notebook 已经成功定位到 ReduceLab 工程目录，并且 CANN SDK 可用。

## 2. 查看工程结构

ReduceLab 工程包含算子实现、Host 侧代码、构建脚本和测试文件。下面通过文件列表了解实验工程的整体组成。

In [ ]:
!find ./src/reduce_lab -maxdepth 3 -type f | sort

文件列表展示了实验的主要组成部分：
- `custom_ops/src/`：3个算子的 Kernel 和 Host 侧实现
- `aclnn_runner/`：Benchmark Runner（用于调用算子并验证结果）
- `scripts/`：编译脚本和测试数据生成脚本

接下来重点查看 Kernel 侧源码，分析输入数据如何完成搬运、局部规约和结果写回。

## 3. 查看 ReduceSum Kernel 源码

ReduceSumLite 的 Kernel 实现位于 `custom_ops/src/ReduceSumLite/op_kernel/reduce_sum_lite.cpp`。核心逻辑包括：

1. 通过 `GetBlockIdx()` 获取当前核编号，确定数据范围
2. 从 GM 逐元素读取输入数据，转换为 float 进行累加
3. 将局部结果写回输出 tensor

In [ ]:
!cat ./src/reduce_lab/custom_ops/src/ReduceSumLite/op_kernel/reduce_sum_lite.cpp | head -60

上述代码展示了 ReduceSum Kernel 的 Init 和 Process 方法。`Init` 绑定 GM 地址，`Process` 执行多核切分和局部规约。

## 4. 编译 ReduceLab

下面执行实验自带的构建脚本。编译过程会完成 Ascend C Kernel 编译、Host 程序构建和必要的链接操作。

首次编译约需 2-3 分钟，请耐心等待。正常情况下，终端最后应输出"算子编译成功"。

In [ ]:
import subprocess

source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && TARGET={TARGET} bash scripts/build_ops.sh',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('算子编译成功')

编译成功后，接下来编译 Benchmark Runner。Runner 是一个 C++ 程序，负责调用 NPU 算子并验证精度。

In [ ]:
source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && source scripts/env_custom_opp.sh && bash scripts/build_runner.sh',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-300:])
else:
    print('Runner 编译成功')

## 5. 生成测试数据

编译完成后，需要生成测试输入数据和 CPU 参考结果。`gen_data.py` 脚本会生成随机输入，并在 CPU 上计算 ReduceSum、ReduceMax 和 TopK 的参考结果，供后续对比验证。

In [ ]:
result = subprocess.run(
    'python3 scripts/gen_data.py --num_tokens 1024 --top_k 4',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True
)
print(result.stdout)
assert os.path.exists(os.path.join(LAB_DIR, 'data/input/x.bin')), '测试数据生成失败'
print('测试数据生成成功')

输出中的 `sum` 和 `max` 是 CPU 参考结果，后续将与 NPU 输出对比。

## 6. 运行 Benchmark 并验证结果

最后执行 Benchmark Runner，它会调用 NPU 算子处理测试数据，并将 NPU 输出与 CPU 参考结果进行比较。若 3 个算子全部 PASS，则说明实验运行成功。

In [ ]:
source_env = f'source {ASCEND_HOME}/set_env.sh'
result = subprocess.run(
    f'{source_env} && source scripts/env_custom_opp.sh && aclnn_runner/build/main_reduce_benchmark data 1024 4',
    shell=True, cwd=LAB_DIR, capture_output=True, text=True, executable='/bin/bash'
)
print(result.stdout)

pass_count = result.stdout.count('PASS')
print(f'\n=== 验证结果: {pass_count}/3 PASS ===')
assert pass_count == 3, f'期望3个PASS，实际{pass_count}个'
print('所有算子验证通过')

输出中的 `result` 是 NPU 计算结果，`ref` 是 CPU 参考结果，`error` 是两者之间的误差。误差为 0 或极小值时显示 PASS。

## 7. 算子源码解读

### ReduceSumLite Kernel 核心逻辑

```cpp
// 多核切分：每个核处理 1/N 的数据（N=核数）
uint32_t blockId = GetBlockIdx();  // 0..N-1
uint32_t start = blockId * blockLength;

// 累加（用 float 避免 FP16 精度损失）
float localSum = 0.0f;
for (uint32_t i = start; i < end; ++i) {
    localSum += static_cast<float>(xGm.GetValue(i));
}

// 写入输出 tensor（不是 workspace！910B D-cache 限制）
yGm.SetValue(blockId, static_cast<half>(localSum));
```

### 平台支持
- **310B**：8 个 AI Core，BLOCK_DIM=8，已验证
- **910B**：20+ 个 AI Core，BLOCK_DIM=20+，已验证
- 算子代码通用，通过 `GetBlockIdx()` 自动适配不同平台

## 实验总结

你已经完成了：
1. 检查实验环境
2. 查看工程结构和 Kernel 源码
3. 编译 3 个 Ascend C 自定义算子
4. 编译 Benchmark Runner
5. 生成测试数据
6. 运行 benchmark 验证精度

**下一步**：打开 `04.03_chapter_test.ipynb` 完成课后实践任务！